# <center>Análisis exploratorio y limpieza de datos del Directorio de la Minería Mexicana.</center>


## Introducción.
Se trabajó con información del Directorio de la Minería Mexicana del Servicio Geológico Mexicano con el objetivo de identificar patrones relacionados con minerales, ubicación geográfica y actividades mineras mediante técnicas de limpieza, transformación y análisis de datos.
### Objetivo del proyecto:
* Realizar procesos de limpieza, transformación y análisis exploratorio de datos utilizando Python y Pandas.
* Integrar SQLite para ejecutar consultas estructuradas y simular un entorno básico de análisis de datos basado en bases de datos relacionales.
* Identificar patrones, frecuencias y distribuciones relevantes dentro del dataset mediante consultas SQL y análisis exploratorio.
* Generar visualizaciones utilizando Matplotlib para facilitar la interpretación de resultados y comunicar hallazgos relevantes.
### Descripción:
El conjunto de datos contiene información relacionada con unidades mineras en México, incluyendo:
* El nombre de la unidad minera.
* Ciudad.
* Estado.
* Minerales explotados.
* Actividades realizadas.
### Preguntas iniciales para análisis:
* ¿Qué estados concentran más actividad minera?
* ¿Qué minerales aparecen con mayor frecuencia?
* ¿Qué actividades mineras son más comunes?

## Problemas detectados en el análisis exploratorio.

Durante el analisis exploratorio inicial se identificó que múltiples registros contenían lista de actividades y minerales dentro de una celda. Para normalizar la información se aplicaron transformaciones mediante **.explode()** en las columnas "realiza" y "mineral", lo que incrementó significativamente la granularidad del dataset. Es decir, la cantidad de registros de ~500 datos aumentó a ~2000, se trabajó inicialmente con este dataset resultante, sin embargo, al realizar la visualización con la *gráfica heatmap* para observar la relación empresas vs actividad se dificultó interpretar los resultados y posibles sobre-interpretaciones en ciertas categorías.

## Justificación del nuevo enfoque.
Debido a que las actividades empresariales representan mejor el perfil operativo y comercial de las organizaciones, se decidió reenfocar el análisis hacia la clasificación de actividades económicas.

Este enfoque permite obtener insights más interpretables y con mayor aplicación para análisis de negocio, segmentación industrial y detección de oportunidades comerciales.

## Explicar la nueva granularidad
En esta versión del analísis la unidad analítica corresponde a:
$$
empresa \times realiza
$$
## Nuevo Analísis exploratoio.
Ahora si se enfocará el estudio para detectar las actividades predominantes de cada estado. Por lo que, se debe realizar la importación de librerías *pandas*, *seaborn*, *sql*, *cv* y *prettytable*.

In [2]:
import pandas as pd
!pip install ipython-sql
!pip install seaborn
import seaborn as sns
%load_ext sql

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
#Leer archivos csv con la librería de pandas
file = "Metalicos_No_Metalicos_Minas.csv"
df = pd.read_csv(file,header=0)

In [4]:
#Visualización inicial del dataset para verificar la correcta carga de información
df.head(5)

,nombre,tipo_unidad,domicilio,asentamiento,ciudad,cp,estado,municipio,no_tel,responsable_puesto,pagina_web,correo_electronico,mineral,realiza,longitud_x,latitud_y,fecha_actualizacion
0,Agnico Eagle Mines Limited,Grupo,"Blvd. Luis Donaldo Colosio n.° 450, nivel 7. C...",Alameda,Hermosillo,83204,Sonora,Hermosillo,662 236 2220,"Amador Montaño Gustavo Ernesto, Director de de...",www.agnicoeagle.com,gustavo.amador@agnicoeagle.com / marco.galindo...,"Oro, plata","Exploración, explotación",-110.987361,29.093361,apr-25
1,"Altos Hornos de México, S.A.B. de C.V.",Grupo,Prol. Juárez s/n. Colonia La Loma,La Loma,Monclova,25770,Coahuila de Zaragoza,Monclova,866 649 3000,"Ancira Elizondo Alfonso, Director general",www.ahmsa.com,ventas@ahmsa.com,Fierro,Comercialización,-101.420389,26.886271,apr-25
2,"ArcelorMittal México, S.A. de C.V.",Grupo,Av. Lázaro Cárdenas n.° 2424. Colonia Residenc...,Residencial San Agustín Primer Sector,San Pedro Garza García,66260,Nuevo León,San Pedro Garza García,811 223 6700,"M. Cairo Víctor, Director ejecutivo",mexico.arcelormittal.com,victor.cairo@arcelormittal.com,Fierro,Fabricación,-100.333539,25.652099,apr-25
3,Argonaut Gold Inc.,Grupo,Blvd. Carlos Quintero Arce n.° 24 sur B. Colon...,Puerta Grande,Hermosillo,83247,Sonora,Hermosillo,662 264 8191,"Young Richard, Presidente CEO",www.argonautgold.com,richasrd.young@argonautgold.com,"Oro, plata",Explotación,-110.995361,29.143750,apr-25
4,Atsa Minerales,Grupo,Vicente Guerrero n.° 6. Colonia San Juan Ixhua...,San Juan Ixhuatepec,Tlalnepantla de Baz,54180,Estado de México,Tlalnepantla de Baz,559 454 0100,"Ricardi Ignacio, Gerente de Administración",atsaminerales.com/,ventas@atsaminerales.com,"Caolín, carbonato de calcio",Transformación de minerales,-99.104110,19.513111,oct-25


Recordar que las variables relevantes para este estudio son estás:
| Nombre del dato | Tipo de dato |
| :--- | :---: |
| nombre | TEXT |
| tipo_unidad | TEXT |
| estado | TEXT |
| mineral | TEXT |
| realiza | TEXT |

In [5]:
columnas_interes = ["nombre","tipo_unidad","estado","mineral","realiza"]
df_1 = df[columnas_interes].copy()
df_1

,nombre,tipo_unidad,estado,mineral,realiza
0,Agnico Eagle Mines Limited,Grupo,Sonora,"Oro, plata","Exploración, explotación"
1,"Altos Hornos de México, S.A.B. de C.V.",Grupo,Coahuila de Zaragoza,Fierro,Comercialización
2,"ArcelorMittal México, S.A. de C.V.",Grupo,Nuevo León,Fierro,Fabricación
3,Argonaut Gold Inc.,Grupo,Sonora,"Oro, plata",Explotación
4,Atsa Minerales,Grupo,Estado de México,"Caolín, carbonato de calcio",Transformación de minerales
...,...,...,...,...,...
574,Unidad La Ciénega,Planta,Durango,"Oro, plata, plomo, zinc","Concentración, flotación, lixiviación, moliend..."
575,Unidad Planta de Cal,Planta,Sonora,Cal viva,Procesamiento de minerales
576,Unidad Velardeña,Planta,Durango,"Cobre, oro, plata, plomo, zinc","Concentración, flotación, molienda, procesamie..."
577,X-Nohbotún,Planta,Yucatán,Agregados pétreos,"Comercialización, distribución de material de ..."


In [6]:
### Normalización de este dataset por "realiza".
df_1["realiza"] = df_1["realiza"].str.split(',')
df_normalizado = df_1.explode("realiza").copy()
df_normalizado

,nombre,tipo_unidad,estado,mineral,realiza
0,Agnico Eagle Mines Limited,Grupo,Sonora,"Oro, plata",Exploración
0,Agnico Eagle Mines Limited,Grupo,Sonora,"Oro, plata",explotación
1,"Altos Hornos de México, S.A.B. de C.V.",Grupo,Coahuila de Zaragoza,Fierro,Comercialización
2,"ArcelorMittal México, S.A. de C.V.",Grupo,Nuevo León,Fierro,Fabricación
3,Argonaut Gold Inc.,Grupo,Sonora,"Oro, plata",Explotación
...,...,...,...,...,...
577,X-Nohbotún,Planta,Yucatán,Agregados pétreos,Comercialización
577,X-Nohbotún,Planta,Yucatán,Agregados pétreos,distribución de material de construcción
578,X-kanakú,Planta,Yucatán,Agregados pétreos,Extracción
578,X-kanakú,Planta,Yucatán,Agregados pétreos,fabricación


In [7]:
#Se resetean los indices, también se corrigen los espacios y las incosistencias entre minúsculas y mayusculas
df_normalizado = df_normalizado.reset_index(drop=True)
df_normalizado['realiza'] = df_normalizado['realiza'].str.strip().str.lower()


In [30]:
df_normalizado.head(5)

,nombre,tipo_unidad,estado,mineral,realiza
0,Agnico Eagle Mines Limited,Grupo,Sonora,"Oro, plata",exploración
1,Agnico Eagle Mines Limited,Grupo,Sonora,"Oro, plata",explotación
2,"Altos Hornos de México, S.A.B. de C.V.",Grupo,Coahuila de Zaragoza,Fierro,comercialización
3,"ArcelorMittal México, S.A. de C.V.",Grupo,Nuevo León,Fierro,fabricación
4,Argonaut Gold Inc.,Grupo,Sonora,"Oro, plata",explotación


In [8]:
df_normalizado.isnull().sum()

nombre         0
tipo_unidad    0
estado         0
mineral        0
realiza        0
dtype: int64

In [9]:
df_normalizado.duplicated().sum()

np.int64(0)

In [10]:
df_normalizado.dtypes

nombre         object
tipo_unidad    object
estado         object
mineral        object
realiza        object
dtype: object

Para utilizar SQL para generar la base de datos apartir del Directorio de Minería. Esto con el fin de fácilitar el análisis de resultados, además de simular el entorno de una base de datos real.

In [11]:
import csv, sqlite3
con = sqlite3.connect("mineria.db")
cur = con.cursor()

In [12]:
#Ademas se carga el modulo SQL magic.
%sql sqlite:///mineria.db

In [13]:
cur.execute('''
    CREATE TABLE IF NOT EXISTS empresas (
        id_empresa INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT,
        tipo_unidad TEXT,
        estado TEXT,
        mineral TEXT,
        realiza TEXT
    )
''')

In [14]:
#Se establece una conección entre SQL magic y la base de datos "mina.db"
df_normalizado.to_sql("empresas", con, if_exists='replace', index=False)

1186

In [15]:
resultado = pd.read_sql_query("SELECT * FROM empresas LIMIT 5", con)
print(resultado)

                                   nombre tipo_unidad                estado  \
0              Agnico Eagle Mines Limited       Grupo                Sonora   
1              Agnico Eagle Mines Limited       Grupo                Sonora   
2  Altos Hornos de México, S.A.B. de C.V.       Grupo  Coahuila de Zaragoza   
3      ArcelorMittal México, S.A. de C.V.       Grupo            Nuevo León   
4                      Argonaut Gold Inc.       Grupo                Sonora   

      mineral           realiza  
0  Oro, plata       exploración  
1  Oro, plata       explotación  
2      Fierro  comercialización  
3      Fierro       fabricación  
4  Oro, plata       explotación  


In [16]:
# Instalar librerías 'ipython-sql' y 'prettytable' usando pip
!pip install ipython-sql prettytable

# Importa la biblioteca 'prettytable', que se utiliza para mostrar datos en una tabla formateada.
import prettytable

# Establezca el formato de visualización predeterminado para prettytable en 'DEFAULT' (es decir, un formato de tabla simple).
prettytable.DEFAULT = 'DEFAULT'

Defaulting to user installation because normal site-packages is not writeable


In [18]:
%%sql
SELECT COUNT(*) FROM empresas;

 * sqlite:///mineria.db
Done.


COUNT(*)
1186


En esta tabla se identificó la coexistencia de minerales metálicos y no metálicos, los cuales pertenecen a cadenas productivas y procesos industriales distintos.

Debido a ello, se decidió enfocar el análisis en minerales metálicos asociados a procesos metalúrgicos, así como realizar una clasificación de actividades para reducir la complejidad categórica y mejorar la interpretabilidad de las visualizaciones.

Como resultado, se generó una tabla denominada clasificacion_actividades, orientada al análisis estratégico de operaciones metalúrgicas.

Primero se realizó una tabla que solo contengan minerales metálicos, através de view y el filtrado con WHERE, esto de la siguiente manera. 

In [19]:
%%sql
DROP VIEW IF EXISTS empresas_metalicas;

 * sqlite:///mineria.db
Done.


[]

In [20]:
%%sql
CREATE VIEW empresas_metalicas AS
SELECT *
FROM empresas
WHERE mineral LIKE '%oro%'
          OR mineral LIKE '%plata%'
          OR mineral LIKE '%cobre%'
          OR mineral LIKE '%zinc%'
          OR mineral LIKE '%plomo%'
          OR mineral LIKE '%molibdeno%'
          OR mineral LIKE '%litio%'
          OR mineral LIKE '%cobalto%'
          OR mineral LIKE '%manganeso%'
          OR mineral LIKE '%hierro%'
          OR mineral LIKE '%fierro%'

 * sqlite:///mineria.db
Done.


[]

In [21]:
%%sql
SELECT COUNT(*) FROM empresas_metalicas;

 * sqlite:///mineria.db
Done.


COUNT(*)
694


In [22]:
%%sql
SELECT nombre, estado, realiza, COUNT(DISTINCT nombre) AS Total
FROM empresas_metalicas
GROUP BY estado, realiza
ORDER BY estado, COUNT(*) DESC
LIMIT 10;

 * sqlite:///mineria.db
Done.


nombre,estado,realiza,Total
Santa Francisca,Aguascalientes,flotación,1
Mina Santa Francisca,Aguascalientes,extracción,1
Mina Santa Francisca,Aguascalientes,explotación,1
Mina Santa Francisca,Aguascalientes,exploración,1
Mina Santa Francisca,Aguascalientes,desarrollo minero,1
"Minera Real de Ángeles, S.A. de C.V. (Unidad Asientos)",Aguascalientes,comercialización,1
"Chapultepec Mining Corporation, S.A. de C.V.",Baja California,extracción,2
"Chapultepec Mining Corporation, S.A. de C.V.",Baja California,exploración,2
Unidad San Felipe,Baja California,explotación,1
"Minera y Metalúrgica del Boleo, S.A.P.I de C.V.",Baja California Sur,explotación,1


Esta última tabla responde ¿Cuántas empresas realizan cada actividad por estado? sin embargo, se puede ver un grado de granularidad bastante alto, y no se puede visualizar de mejor manera la diversificación de cada estado. Es necesario realizar una clasificación de actividades metalurgicas.  

Luego se realiza una tabla denominada claficación de actividades

In [23]:
%%sql
DROP VIEW IF EXISTS clasificacion_actividades;

 * sqlite:///mineria.db
Done.


[]

In [24]:
%%sql
CREATE VIEW clasificacion_actividades AS
SELECT DISTINCT
    realiza,

    CASE
        WHEN realiza IN (
            'exploración',
            'extracción',
            'explotación',
            'desarrollo minero'
        ) THEN 'extractiva'

        WHEN realiza IN (
            'molienda',
            'flotación',
            'lixiviación',
            'concentración',
            'trituración de materiales',
            'cribado',
            'clasificación',
            'secado'
        ) THEN 'procesamiento'

        WHEN realiza IN (
            'beneficio (refinamiento)',
            'hidrometalurgia',
            'calcinación'
        ) THEN 'refinamiento'

        WHEN realiza IN (
            'comercialización',
            'compra',
            'distribución de material de construcción'
        ) THEN 'comercial'

        WHEN realiza IN (
            'fabricación',
            'transformación de minerales',
            'transformación (mecánica o química)'
        ) THEN 'industrial'

        ELSE 'otros'

    END AS categoria_actividad
FROM empresas_metalicas;

 * sqlite:///mineria.db
Done.


[]

In [25]:
%%sql
SELECT estado, categoria_actividad, COUNT(DISTINCT nombre) AS Total
FROM empresas_metalicas e
JOIN clasificacion_actividades c
ON c.realiza = e.realiza
GROUP BY estado,categoria_actividad
ORDER BY Total DESC;

 * sqlite:///mineria.db
Done.


estado,categoria_actividad,Total
Sonora,extractiva,69
Durango,extractiva,57
Durango,comercial,32
Durango,procesamiento,20
Sonora,comercial,17
Chihuahua,extractiva,16
Sonora,procesamiento,16
Hidalgo,extractiva,15
Durango,otros,10
Ciudad de México,extractiva,9


Esta consulta permite visualizar los tipos de actividades registradas por estado dentro del dataset.

Aunque la Ciudad de México presenta una cantidad considerable de unidades clasificadas en actividades extractivas, es posible que dichas unidades correspondan principalmente a corporativos, grupos empresariales u oficinas administrativas, y no necesariamente a operaciones mineras físicas.

Por ello, se propone complementar el análisis mediante un conteo por tipo de unidad registrada.


In [62]:
%%sql
SELECT 
    estado,
    tipo_unidad,
    COUNT(DISTINCT nombre) AS registros
FROM empresas_metalicas 
WHERE tipo_unidad IN ("Empresa","Grupo")
GROUP BY estado, tipo_unidad
ORDER BY estado, registros DESC

 * sqlite:///mineria.db
Done.


estado,tipo_unidad,registros
Aguascalientes,Empresa,1
Baja California,Empresa,1
Baja California Sur,Empresa,1
Chihuahua,Empresa,12
Chihuahua,Grupo,1
Ciudad de México,Empresa,8
Ciudad de México,Grupo,6
Coahuila de Zaragoza,Empresa,5
Coahuila de Zaragoza,Grupo,1
Colima,Empresa,3


In [63]:
%%sql
SELECT 
    estado,
    tipo_unidad,
    COUNT(DISTINCT nombre) AS registros
FROM empresas_metalicas 
WHERE tipo_unidad IN ("Mina","Planta")
GROUP BY estado, tipo_unidad
ORDER BY estado, registros DESC

 * sqlite:///mineria.db
Done.


estado,tipo_unidad,registros
Aguascalientes,Planta,1
Aguascalientes,Mina,1
Baja California,Mina,1
Chihuahua,Mina,3
Coahuila de Zaragoza,Planta,2
Colima,Mina,2
Colima,Planta,1
Durango,Mina,33
Durango,Planta,21
Guanajuato,Mina,2


Con el fin de validar la naturaleza de las actividades registradas por estado, se realizó una segmentación por tipo de unidad.

Los resultados muestran que la Ciudad de México concentra principalmente registros correspondientes a empresas y grupos corporativos, mientras que no presenta registros asociados a minas o plantas operativas dentro del dataset analizado.

Esto sugiere que la participación de la entidad en categorías extractivas se relaciona principalmente con funciones corporativas, administrativas o de gestión empresarial, y no necesariamente con operaciones mineras físicas localizadas en la región.

En contraste, estados como Sonora y Durango presentan una mayor cantidad de unidades clasificadas como minas y plantas, lo cual refleja una presencia operativa más representativa de la actividad minera y metalúrgica.

Ahora debemos hacer el dashbord, utilizaré Power Bi para visualizar de mejor manera los datos.

In [65]:
query_1 = """
SELECT estado, categoria_actividad, COUNT(DISTINCT nombre) AS Total
FROM empresas_metalicas e
JOIN clasificacion_actividades c
ON c.realiza = e.realiza
GROUP BY estado,categoria_actividad
ORDER BY estado, Total DESC;
"""
df_actividades = pd.read_sql_query(query_1, con)

In [67]:
query_2 = """
SELECT 
    estado,
    tipo_unidad,
    COUNT(DISTINCT nombre) AS registros
FROM empresas_metalicas 
WHERE tipo_unidad IN ("Empresa","Grupo")
GROUP BY estado, tipo_unidad
ORDER BY estado, registros DESC
"""
df_corporativos = pd.read_sql_query(query_2, con)

In [68]:
query_3 = """
SELECT 
    estado,
    tipo_unidad,
    COUNT(DISTINCT nombre) AS registros
FROM empresas_metalicas 
WHERE tipo_unidad IN ("Mina","Planta")
GROUP BY estado, tipo_unidad
ORDER BY estado, registros DESC
"""
df_operativas = pd.read_sql_query(query_3, con)

In [69]:
df_actividades.to_csv("actividades.csv", index=False)
df_corporativos.to_csv("corporativos.csv", index=False)
df_operativas.to_csv("operativas.csv", index=False)